# 04 — Financial Inclusion EDA & Demographic Gap (Orang 1)

**Owner:** rrrafaa | **Data:** Global Findex Indonesia 2024

**Research questions:** How prevalent is account ownership? How does it differ by income, gender, education, age, and rural/urban category? How large are the weighted group gaps?

**Approach:** survey-weighted descriptive EDA, no causal inference. 2024 is a cross-section, not a time series. *Do not publish respondent-level data.*


## 1. Environment and private input

This notebook independently loads original ZIP; it does not depend on runtime state in notebook 01. On Colab, upload the ZIP. Locally, use data/raw/.


In [ ]:
from pathlib import Path
import hashlib,io,zipfile,sys,subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
REPO="Indonesia-Financial-Inclusion-Digital-Finance-Analytics"
ZIP="IDN_2024_FINDEX_v02_M_CSV.zip"
CSV="Findex_Microdata_2025_updateIndonesia.csv"
SHA="fc50f0a843e289a3b9784ac81dce1018a8195388608575e6c8315c195073a64c"
root=next((p for p in [Path.cwd(),*Path.cwd().parents,Path("/content")/REPO] if (p/"data/raw/README.md").exists()),None)
if root is None and "google.colab" in sys.modules:
    root=Path("/content")/REPO
    subprocess.run(["git","clone","--depth","1","https://github.com/rrrafaa/"+REPO+".git",str(root)],check=True)
if root is None: raise FileNotFoundError("Run Jupyter from the cloned repo")
raw=root/"data/raw";out=root/"data/processed"
raw.mkdir(parents=True,exist_ok=True);out.mkdir(parents=True,exist_ok=True)
p=next((p for p in [raw/ZIP,Path("/content")/ZIP,Path.cwd()/ZIP] if p.exists()),None)
if p is None and "google.colab" in sys.modules:
    from google.colab import files
    uploaded=files.upload()
    if ZIP not in uploaded: raise FileNotFoundError("Expected original ZIP "+ZIP)
    p=raw/ZIP;p.write_bytes(uploaded[ZIP])
if p is None: raise FileNotFoundError("Place original ZIP in data/raw/"+ZIP)
assert hashlib.sha256(p.read_bytes()).hexdigest()==SHA,"Wrong source ZIP checksum"
with zipfile.ZipFile(p) as z: df=pd.read_csv(io.BytesIO(z.read(CSV)),low_memory=False)
assert df.shape==(1068,199)
assert df.wgt.notna().all() and df.wgt.gt(0).all()
print("Ready:",df.shape)



## 2. Derived categories and validity rules

The original dataset already contains core 0/1 financial indicators. Keep unknown answers as missing; preserve the original data unchanged.


In [ ]:
d=df.copy()
d["gender"]=d.female.map({1:"Female",2:"Male"})
d["income_quintile"]=d.inc_q.map({1:"Q1 - lowest",2:"Q2",3:"Q3",4:"Q4",5:"Q5 - highest"})
d["education"]=d.educ.map({1:"Primary or less",2:"Secondary",3:"Tertiary or more"})
d["residence"]=d.urbanicity.map({1:"Rural",2:"Urban"})
d["age_group"]=pd.cut(d.age,bins=[14,24,34,44,54,np.inf],labels=["15-24","25-34","35-44","45-54","55+"])
INDICATORS={"account":"Owns any account","account_fin":"Financial institution account","account_mob":"Mobile money account","anydigpayment":"Used digital payment","saved":"Saved money","borrowed":"Borrowed money"}
for key in INDICATORS:
    assert d[key].dropna().isin([0,1]).all(),f"Unexpected code in {key}"
display(d[["gender","income_quintile","education","residence","age_group"]].head())


## 3. Weighted metrics helper

Use **sum(w × indicator) / sum(w)** for valid respondents only. Record unweighted n to flag unstable groups. Compare descriptive gaps in *percentage points*.


In [ ]:
def weighted_share(frame,indicator):
    sample=frame.loc[frame[indicator].isin([0,1]) & frame.wgt.gt(0),[indicator,"wgt"]]
    if sample.empty: return (np.nan,0,0.0)
    numerator=(sample[indicator]*sample.wgt).sum()
    denominator=sample.wgt.sum()
    return (100*numerator/denominator,len(sample),float(denominator))

def by_group(group_col,indicator="account"):
    rows=[]
    for key,part in d.groupby(group_col,observed=True,dropna=False):
        rate,n,weight_total=weighted_share(part,indicator)
        rows.append({"dimension":group_col,"group":str(key),"metric":indicator,"rate_pct":round(rate,3),"n_valid":n,"weight_total":weight_total,"small_n_warning":n<30})
    return pd.DataFrame(rows)

overview=pd.DataFrame([{"metric":key,"label":label,"rate_pct":weighted_share(d,key)[0],"n_valid":weighted_share(d,key)[1]} for key,label in INDICATORS.items()])
display(overview.round(2))
assert 50 < overview.loc[overview.metric.eq("account"),"rate_pct"].iloc[0] < 65,"Investigate unexpected national account rate"


## 4. National headline KPIs


In [ ]:
fig,ax=plt.subplots(figsize=(10,4.5))
tmp=overview.sort_values("rate_pct")
ax.barh(tmp.label,tmp.rate_pct)
ax.set_xlim(0,100);ax.set_xlabel("Weighted respondent percentage (%)")
ax.set_title("Indonesia 2024 — Financial inclusion overview")
for i,v in enumerate(tmp.rate_pct): ax.text(v+1,i,f"{v:.1f}%",va="center")
plt.tight_layout();plt.show()


## 5. Account ownership by income quintile

Check the difference Q5 − Q1. Keep income-quintile ordering.


In [ ]:
income=by_group("income_quintile")
display(income)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(income.group,income.rate_pct)
ax.set(ylim=(0,100),ylabel="Weighted account ownership (%)",title="Account ownership by income quintile")
plt.xticks(rotation=15);plt.tight_layout();plt.show()
q1=income.loc[income.group.eq("Q1 - lowest"),"rate_pct"].iloc[0]
q5=income.loc[income.group.eq("Q5 - highest"),"rate_pct"].iloc[0]
print(f"Q5 minus Q1 account ownership gap: {q5-q1:.1f} percentage points")


## 6. Account ownership by gender, education, age group and residence


In [ ]:
dimensions=["gender","education","age_group","residence"]
tables=[]
fig,axes=plt.subplots(2,2,figsize=(13,9))
for ax,dim in zip(axes.flat,dimensions):
    t=by_group(dim);tables.append(t)
    ax.barh(t.group,t.rate_pct)
    ax.set_xlim(0,100);ax.set_title("Account ownership — "+dim.replace("_"," ").title())
    ax.set_xlabel("Weighted percentage (%)")
    for i,v in enumerate(t.rate_pct):ax.text(min(v+1,97),i,f"{v:.1f}%",va="center",fontsize=9)
plt.tight_layout();plt.show()
segments=pd.concat([income,*tables],ignore_index=True)
display(segments)
print("Segments with unweighted n < 30:")
display(segments.loc[segments.small_n_warning])


## 7. Multiple indicators by gender and income

`account_fin`, `account_mob`, and `anydigpayment` may overlap. Do not stack them to 100%.


In [ ]:
cross=pd.concat([by_group(dim,indicator) for dim in ["gender","income_quintile","residence"] for indicator in INDICATORS],ignore_index=True)
display(cross.head(20))
pivot=cross.query("dimension=='gender'").pivot(index="metric",columns="group",values="rate_pct")
display(pivot.round(1))
pivot.plot(kind="bar",figsize=(10,5))
plt.ylabel("Weighted percentage (%)");plt.title("Financial outcomes by gender")
plt.xticks(rotation=30,ha="right");plt.ylim(0,100);plt.tight_layout();plt.show()


## 8. Account gap table and simple data validation


In [ ]:
gap_rows=[]
for dim in dimensions+["income_quintile"]:
    t=by_group(dim)
    valid=t.loc[~t.small_n_warning & t.rate_pct.notna()]
    if len(valid)<2: continue
    hi=valid.loc[valid.rate_pct.idxmax()]
    lo=valid.loc[valid.rate_pct.idxmin()]
    gap_rows.append({"dimension":dim,"highest_group":hi.group,"lowest_group":lo.group,"max_rate_pct":hi.rate_pct,"min_rate_pct":lo.rate_pct,"gap_pp":round(hi.rate_pct-lo.rate_pct,3)})
gaps=pd.DataFrame(gap_rows).sort_values("gap_pp",ascending=False)
display(gaps)
national=weighted_share(d,"account")[0]
print(f"National weighted account rate: {national:.2f}%")
print("Dataset coverage:",len(d),"survey respondents, year:",d.year.unique())
assert np.isclose(national,overview.loc[overview.metric.eq("account"),"rate_pct"].iloc[0])
assert not segments.rate_pct.dropna().gt(100).any()


## 9. Save dashboard-ready aggregates LOCALLY

Files in `data/processed/` are ignored by Git. Use aggregated metrics only when connecting Google Sheets or Looker Studio. Review small cells before any publication.


In [ ]:
overview.round(4).to_csv(out/"person1_overview_kpis.csv",index=False)
segments.to_csv(out/"person1_account_segments.csv",index=False)
cross.to_csv(out/"person1_cross_metrics.csv",index=False)
gaps.to_csv(out/"person1_gap_summary.csv",index=False)
print("Wrote 4 aggregate CSVs to",out)
print("Do not git add raw microdata or individual-level data.")


## 10. Interpretation template and QA handoff

Write your final insights after examining the results:

- **Headline:** national weighted account ownership is __%.
- **Income gap:** Q5 − Q1 is __ percentage points.
- **Largest descriptive gap:** __.
- **Quality caveat:** subgroups with `small_n_warning=True` should not be interpreted confidently.
- **Policy hypothesis:** identify access challenges to investigate; never claim that income or internet *caused* an outcome using this cross-sectional dataset.

### Acceptance checklist
- [ ] Run all cells on the verified input.
- [ ] Compare national metrics against official Findex definitions.
- [ ] Confirm mapping with original codebook and Person 2.
- [ ] Review warnings, labels, denominator rules.
- [ ] Submit notebook through PR without raw CSV or ZIP.
- [ ] Prepare Page 1 Overview and Page 2 Inclusion Gap in Looker Studio.
